# Лабораторна робота №2
## Частина 1

Мета роботи: ознайомитись з підготовчим етапом науки про дані, завантаженням даних, їх очищенням та аналізом за допомогою бібліотеки pandas.

In [2]:
# Імпорт необхідних бібліотек

import pandas as pd
import urllib.request
import os
from datetime import datetime
import glob

### Завдання 1

Створити функцію для завантаження VHI-даних для кожної області України з сайту NOAA.
При збереженні файлу до імені потрібно додати дату та час завантаження.
Передбачити механізм уникнення повторного завантаження файлів.

In [7]:
def download_vhi_data(province_id):
    
    # Формування посилання для завантаження даних
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    # Папка для збереження файлів
    folder = "data"
    
    # Якщо папки не існує — створити
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    # Перевірка чи файл для області вже існує
    existing_files = glob.glob(f"{folder}/vhi_{province_id}_*.csv")
    
    if existing_files:
        print(f"Дані для області {province_id} вже завантажені.")
        return
    
    # Формування імені файлу з датою та часом
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{folder}/vhi_{province_id}_{timestamp}.csv"
    
    # Завантаження файлу
    urllib.request.urlretrieve(url, filename)
    
    print(f"Файл успішно завантажено: {filename}")

### Завдання 2

Завантажити дані для всіх адміністративних областей України.

In [9]:
for province in range(1, 28):
    download_vhi_data(province)

Файл успішно завантажено: data/vhi_1_20260309_154011.csv
Файл успішно завантажено: data/vhi_2_20260309_154016.csv
Файл успішно завантажено: data/vhi_3_20260309_154019.csv
Файл успішно завантажено: data/vhi_4_20260309_154021.csv
Файл успішно завантажено: data/vhi_5_20260309_154024.csv
Файл успішно завантажено: data/vhi_6_20260309_154025.csv
Файл успішно завантажено: data/vhi_7_20260309_154027.csv
Файл успішно завантажено: data/vhi_8_20260309_154030.csv
Файл успішно завантажено: data/vhi_9_20260309_154033.csv
Файл успішно завантажено: data/vhi_10_20260309_154034.csv
Файл успішно завантажено: data/vhi_11_20260309_154036.csv
Файл успішно завантажено: data/vhi_12_20260309_154038.csv
Файл успішно завантажено: data/vhi_13_20260309_154040.csv
Файл успішно завантажено: data/vhi_14_20260309_154041.csv
Файл успішно завантажено: data/vhi_15_20260309_154043.csv
Файл успішно завантажено: data/vhi_16_20260309_154045.csv
Файл успішно завантажено: data/vhi_17_20260309_154047.csv
Файл успішно завантажен

### Завдання 3

Зчитати завантажені файли у pandas DataFrame.

In [66]:
def load_vhi_data():

    files = glob.glob("data/*.csv")

    dataframes = []

    for file in files:

        # читаем файл
        df = pd.read_csv(file, skiprows=2, header=None)

        # удаляем последнюю пустую колонку
        df = df.iloc[:, :7]

        # задаем названия колонок
        df.columns = ["year","week","SMN","SMT","VCI","TCI","VHI"]

        # получаем province_id из имени файла
        filename = os.path.basename(file)
        province_id = int(filename.split("_")[1])

        df["province_id"] = province_id

        dataframes.append(df)

    combined_df = pd.concat(dataframes, ignore_index=True)

    print("Дані успішно завантажені")

    return combined_df

In [67]:
df = load_vhi_data()

df.head()

Дані успішно завантажені


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,<tt><pre>1982,1.0,0.059,258.24,51.11,48.78,49.95,10
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,10
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,10
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,10
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,10


### Завдання 4

Здійснити очищення даних (Data Cleaning):
- прибрати пропущені значення
- видалити зайвий текст
- перейменувати стовпці

In [78]:
def clean_vhi_data(df):

    # Видалення HTML-тегів з колонки year
    df["year"] = df["year"].astype(str)
    df["year"] = df["year"].str.replace("<tt><pre>", "", regex=False)
    df["year"] = df["year"].str.replace("</pre></tt>", "", regex=False)

    # Видалення порожніх значень у колонці year
    df = df[df["year"] != ""]

    # Перетворення типів даних
    df["year"] = df["year"].astype(int)
    df["week"] = df["week"].astype(int)

    # Заповнення пропущених значень
    df = df.fillna(0)

    # Перейменування колонок
    df.columns = ["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI", "province_id"]

    print("Очищення даних завершено")
    print("Кількість записів:", len(df))

    return df

In [79]:
df = clean_vhi_data(df)

df.head()

Очищення даних завершено
Кількість записів: 60372


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,1,0.059,258.24,51.11,48.78,49.95,10
1,1982,2,0.063,261.53,55.89,38.20,47.04,10
2,1982,3,0.063,263.45,57.30,32.69,44.99,10
3,1982,4,0.061,265.10,53.96,28.62,41.29,10
4,1982,5,0.058,266.42,46.87,28.57,37.72,10


### Завдання 5

Реалізувати процедуру зміни індексів областей.

У даних NOAA області індексуються за англійською абеткою
(Province 1 — Cherkasy). Необхідно змінити індекси так,
щоб області індексувались за українською абеткою
(1 — Вінницька область).

In [84]:
def change_province_index(df):

    mapping = {
        1:22, 2:24, 3:23, 4:25, 5:3,
        6:4, 7:8, 8:9, 9:10, 10:11,
        11:12, 12:13, 13:14, 14:15,
        15:16, 16:17, 17:18, 18:19,
        19:20, 20:21, 21:5, 22:6,
        23:7, 24:1, 25:2, 26:26, 27:27
    }

    df["province_id"] = df["province_id"].replace(mapping)

    print("Індекси областей змінено")

    return df

In [96]:
df = change_province_index(df)
df.head()

Індекси областей змінено


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,1,0.059,258.24,51.11,48.78,49.95,14
1,1982,2,0.063,261.53,55.89,38.20,47.04,14
2,1982,3,0.063,263.45,57.30,32.69,44.99,14
3,1982,4,0.061,265.10,53.96,28.62,41.29,14
4,1982,5,0.058,266.42,46.87,28.57,37.72,14


### Завдання 6

Реалізувати вибірку: ряд значень VHI для вказаної області за певний рік.

In [80]:
def get_vhi_for_year(df, province_id, year):
    
    # Фільтрація даних
    result = df[(df["province_id"] == province_id) & (df["year"] == year)]
    
    print("Результат вибірки:")
    
    return result

In [81]:
get_vhi_for_year(df, 5, 2000)

Результат вибірки:


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
50150,2000,1,0.028,257.51,28.08,49.96,39.02,5
50151,2000,2,0.027,255.96,29.87,54.62,42.24,5
50152,2000,3,0.028,254.40,30.93,60.39,45.66,5
50153,2000,4,0.030,253.99,31.84,62.93,47.38,5
50154,2000,5,0.030,253.33,29.91,68.01,48.96,5
50155,2000,6,0.030,253.30,25.52,71.71,48.61,5
50156,2000,7,0.030,254.96,20.90,71.75,46.32,5
50157,2000,8,0.032,257.12,19.80,71.77,45.79,5
50158,2000,9,0.035,259.48,18.63,72.44,45.54,5
50159,2000,10,0.040,261.76,17.10,74.40,45.75,5


### Завдання 7

Реалізувати вибірку: ряд значень VHI за вказаний діапазон років
для вибраних областей.

In [87]:
def get_vhi_range(df, provinces, start_year, end_year):

    result = df[
        (df["province_id"].isin(provinces)) &
        (df["year"] >= start_year) &
        (df["year"] <= end_year)
    ]

    print("Результат вибірки:")
    display(result)

    return result

In [88]:
get_vhi_range(df, [5, 10], 1995, 2000)

Результат вибірки:


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
27520,1995,1,-1.000,-1.00,-1.00,-1.00,-1.00,5
27521,1995,2,-1.000,-1.00,-1.00,-1.00,-1.00,5
27522,1995,3,-1.000,-1.00,-1.00,-1.00,-1.00,5
27523,1995,4,0.034,249.71,25.78,78.24,52.01,5
27524,1995,5,0.034,253.03,27.32,70.31,48.82,5
...,...,...,...,...,...,...,...,...
59145,2000,48,0.054,274.69,14.90,28.38,21.64,10
59146,2000,49,0.049,274.22,15.27,22.53,18.90,10
59147,2000,50,0.046,273.64,15.61,18.19,16.90,10
59148,2000,51,0.046,273.26,17.47,14.77,16.12,10


,year,week,SMN,SMT,VCI,TCI,VHI,province_id
27520,1995,1,-1.000,-1.00,-1.00,-1.00,-1.00,5
27521,1995,2,-1.000,-1.00,-1.00,-1.00,-1.00,5
27522,1995,3,-1.000,-1.00,-1.00,-1.00,-1.00,5
27523,1995,4,0.034,249.71,25.78,78.24,52.01,5
27524,1995,5,0.034,253.03,27.32,70.31,48.82,5
...,...,...,...,...,...,...,...,...
59145,2000,48,0.054,274.69,14.90,28.38,21.64,10
59146,2000,49,0.049,274.22,15.27,22.53,18.90,10
59147,2000,50,0.046,273.64,15.61,18.19,16.90,10
59148,2000,51,0.046,273.26,17.47,14.77,16.12,10


### Завдання 8

Знайти екстремальні значення (мінімум, максимум), а також середнє значення та медіану для вибраних областей.

In [82]:
def vhi_statistics(df, provinces):
    
    data = df[df["province_id"].isin(provinces)]
    
    print("Мінімальне значення VHI:", data["VHI"].min())
    print("Максимальне значення VHI:", data["VHI"].max())
    print("Середнє значення VHI:", data["VHI"].mean())
    print("Медіана VHI:", data["VHI"].median())

In [97]:
vhi_statistics(df, [3,5,10])

Мінімальне значення VHI: -1.0
Максимальне значення VHI: 85.14
Середнє значення VHI: 46.666440071556345
Медіана VHI: 47.41
